# Global LM head: three models, three math datasets

**T4 GPU, Internet on, Run All.** One T4 is used; no attachments are needed.

Models: GPT-2 (released CODI weights), SmolLM2-135M, Qwen2.5-0.5B.
Evaluation: GSM8K, SVAMP, ASDiv's single-number-answer subset.

Only the model and evaluation dataset vary. Everything else keeps the earlier head
experiment: **eager PyTorch, FP16, batch 1, dense versus rank 96**, seed 89, the same
head initialization/distillation/recovery, 16 timing questions x 3 repeats, token caps
64 CODI / 256 explicit. There is no transformer-optimization or rank/backend sweep.

All heads are fitted on the same GSM8K training split, then reused across datasets.
Each model/dataset pair has four mean timing tables, clean latency and full eligible-set
accuracy. Expand its result panel to see the tables; no CSV downloads are needed.

The released CODI checkpoint supports GPT-2. The other two models run explicit CoT
from their published base checkpoints; **their CODI columns are N/A**. No untrained
latent loop is presented as CODI. Their absolute math accuracy is not a controlled
comparison with the math-trained GPT-2 checkpoint; compare each model's dense and
compressed head. Training new CODI backbones is not part of this notebook.

In [ ]:
import gc,gzip,hashlib,importlib.util,json,logging,os,pathlib,random,subprocess,sys,time,warnings
from contextlib import contextmanager
from urllib.request import urlopen

# Fixed experiment defaults: no configuration is required.
SEED=89
MODES=['codi','explicit_cot']
FIT_QUESTIONS,SELECT_QUESTIONS,RECOVERY_QUESTIONS=1024,256,256
MAX_FIT_STATES,MAX_SELECT_STATES,MAX_RECOVERY_STATES=4096,1024,2048
CLEAN_EPOCHS,RECOVERY_EPOCHS=4,2
DISTILL_BATCH_SIZE=8
COLLECT_BATCH_SIZE=8
RANKS=(32,64,96)
MAX_NEW_TOKENS={'codi':64,'explicit_cot':256}
TIMING_QUESTIONS,TIMING_REPEATS=16,3
OUTPUT_ROOT=pathlib.Path('/kaggle/working/codi_model_dataset_ablations')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
BOOTSTRAP_TIMES=[]
@contextmanager
def bootstrap_stage(name):
    start=time.perf_counter()
    try: yield
    finally: BOOTSTRAP_TIMES.append(dict(name=name,cpu_wall_ms=1000*(time.perf_counter()-start)))

# Do not reinstall datasets/pandas/dill or change Kaggle's PyTorch/CUDA.
# GSM8K is read directly from its canonical JSONL files.
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ.pop('HF_HUB_DISABLE_XET',None)
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT','300')
REPO_URL='https://github.com/0x0shephard/latent-reasoning.git'
BASE_COMMIT='6a8d2e61950c67f012d0a9ba13ec8a70f3a25019'
REPO_DIR='/kaggle/working/latent-reasoning'
setup_log=OUTPUT_ROOT/'setup.log'
def checked(command):
    result=subprocess.run(command,capture_output=True,text=True)
    with setup_log.open('a') as log: log.write(result.stdout+'\n'+result.stderr+'\n')
    if result.returncode:
        raise RuntimeError(result.stdout[-3000:]+'\n'+result.stderr[-5000:])
    return result
with bootstrap_stage('git_clone_and_checkout'):
    if not pathlib.Path(REPO_DIR).exists(): checked(['git','clone',REPO_URL,REPO_DIR])
    checked(['git','-C',REPO_DIR,'fetch','origin'])
    checked(['git','-C',REPO_DIR,'checkout','--detach',BASE_COMMIT])
os.chdir(REPO_DIR)
sys.path.insert(0,REPO_DIR)
with bootstrap_stage('dependency_installation'):
    checked([sys.executable,'-m','pip','install','-q','transformers==4.52.4','peft==0.15.2',
             'huggingface_hub>=0.34.0,<1.0','hf_xet','accelerate==1.7.0','pyyaml'])
    probe=subprocess.run([sys.executable,'-c','import peft; from transformers import GPT2LMHeadModel'],capture_output=True,text=True)
    if probe.returncode and 'torchao' in (probe.stdout+probe.stderr):
        checked([sys.executable,'-m','pip','uninstall','-y','torchao'])
    checked([sys.executable,'-c','import torch,peft,huggingface_hub,hf_xet; from transformers import GPT2LMHeadModel,DynamicCache'])
print('Setup ready. Unrelated system-package messages are retained in setup.log; failed installs/imports stop here.')

In [ ]:
RUNTIME_SOURCE = '"""CODI/explicit decoding and an additive, four-column bottleneck report.\n\nThe report partitions a single CUDA-stream timeline, including host-induced idle\nintervals. It is an instrumented elapsed-time breakdown, not summed kernel time.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nfrom dataclasses import dataclass\nimport gzip\nimport json\nfrom pathlib import Path\nimport time\nimport uuid\n\nimport torch\nfrom torch import nn\nfrom src.models.official_codi import official_codi_base_model, _normalized_official_questions\nfrom src.inference.official_codi_fast import FastCODIGeneration\n\n\nclass Timeline:\n    def __init__(self, device=\'cpu\', enabled=True, unified_clock=False):\n        self.device = torch.device(device)\n        self.enabled = enabled\n        self.unified_clock = unified_clock\n        self.context = {}\n        self.records = []\n        self.pending = []\n        self.stack = []\n        self.counter = 0\n        self.trace_id = uuid.uuid4().hex\n\n    def start(self, name, kind=\'stage\', gpu=True, **extra):\n        if not self.enabled:\n            return None\n        self.counter += 1\n        row = dict(self.context, trace_id=self.trace_id, event_id=self.counter,\n                   parent_id=self.stack[-1] if self.stack else None,\n                   name=name, kind=kind, **extra)\n        marker = None if self.unified_clock else torch.profiler.record_function(name)\n        if marker is not None:\n            marker.__enter__()\n        row[\'_wall_start\'] = time.perf_counter()\n        events = None\n        if (gpu or self.unified_clock) and self.device.type == \'cuda\':\n            events = (torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True))\n            events[0].record()\n        self.stack.append(self.counter)\n        return row, events, marker\n\n    def stop(self, token):\n        if token is None:\n            return\n        row, events, marker = token\n        if events:\n            events[1].record()\n        row[\'cpu_wall_ms\'] = (time.perf_counter() - row.pop(\'_wall_start\')) * 1000\n        if marker is not None:\n            marker.__exit__(None, None, None)\n        if self.stack and self.stack[-1] == row[\'event_id\']:\n            self.stack.pop()\n        self.pending.append((row, events))\n\n    @contextmanager\n    def span(self, name, kind=\'stage\', gpu=True, **extra):\n        token = self.start(name, kind, gpu, **extra)\n        try:\n            yield\n        finally:\n            self.stop(token)\n\n    @contextmanager\n    def metadata(self, **values):\n        previous = self.context.copy()\n        self.context.update(values)\n        try:\n            yield\n        finally:\n            self.context = previous\n\n    def resolve(self):\n        if self.device.type == \'cuda\' and self.pending:\n            torch.cuda.synchronize(self.device)\n        for row, events in self.pending:\n            row[\'cuda_stream_ms\'] = events[0].elapsed_time(events[1]) if events else None\n            self.records.append(row)\n        self.pending.clear()\n\n    @contextmanager\n    def modules(self, roots, recursive=True):\n        if not self.enabled:\n            yield\n            return\n        handles, stacks, seen = [], {}, set()\n        for prefix, root in roots:\n            for suffix, module in (root.named_modules() if recursive else [(\'\', root)]):\n                if id(module) in seen:\n                    continue\n                seen.add(id(module))\n                name = prefix + (\'.\' + suffix if suffix else \'\')\n                stacks[id(module)] = []\n                def pre(mod, args, label=name):\n                    token = self.start(label, kind=\'module\', inclusive=True,\n                                       module_type=type(mod).__name__)\n                    stacks[id(mod)].append(token)\n                def post(mod, args, output):\n                    self.stop(stacks[id(mod)].pop())\n                handles.append(module.register_forward_pre_hook(pre))\n                handles.append(module.register_forward_hook(post, always_call=True))\n        try:\n            yield\n        finally:\n            for handle in handles:\n                handle.remove()\n\n    def flush(self, path):\n        self.resolve()\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        with gzip.open(path, \'at\', encoding=\'utf-8\') as handle:\n            for row in self.records:\n                handle.write(json.dumps(row, default=str) + \'\\n\')\n        self.records.clear()\n\n\n@dataclass\nclass PreparedBatch:\n    indices: tuple\n    ids: torch.Tensor\n    mask: torch.Tensor\n\n\ndef prepare_questions(tokenizer, questions, batch_size, timeline=None):\n    timeline = timeline or Timeline(enabled=False)\n    with timeline.span(\'question_normalization\', gpu=False):\n        questions = _normalized_official_questions(questions)\n    with timeline.span(\'question_tokenization\', gpu=False):\n        encoded = tokenizer(questions, add_special_tokens=False, padding=False)[\'input_ids\']\n    result = []\n    for start in range(0, len(encoded), batch_size):\n        part = encoded[start:start+batch_size]\n        with timeline.metadata(batch_index=start//batch_size):\n            with timeline.span(\'cpu_padding_and_tensor_allocation\', gpu=False):\n                width = max(map(len, part))\n                ids = torch.full((len(part), width), tokenizer.pad_token_id, dtype=torch.long)\n                mask = torch.zeros_like(ids)\n                for i, values in enumerate(part):\n                    if not values:\n                        raise ValueError(\'Empty question\')\n                    ids[i, -len(values):] = torch.tensor(values)\n                    mask[i, -len(values):] = 1\n            result.append(PreparedBatch(tuple(range(start, start+len(part))), ids, mask))\n    return result\n\n\ndef select_token(head, hidden, vocabulary_stop):\n    specialized = getattr(head, \'select_token\', None)\n    return specialized(hidden, vocabulary_stop=vocabulary_stop) if callable(specialized) else head(hidden)[..., :vocabulary_stop].argmax(-1)\n\n\n@torch.no_grad()\ndef decode(model, tokenizer, batches, head, *, mode, device, max_new_tokens=256,\n           latent_iterations=6, timeline=None, observer=None, forced_tokens=None):\n    """Same CODI body-only contract; explicit mode starts directly from the question.\n\n    Fixed replay evaluates every head but feeds back dense-reference token IDs.\n    Timed CUDA paths must be called inside torch.inference_mode() by the runner.\n    """\n    if mode not in (\'codi\', \'explicit_cot\'):\n        raise ValueError(mode)\n    if max_new_tokens <= 0:\n        raise ValueError(\'max_new_tokens must be positive\')\n    timeline = timeline or Timeline(device, enabled=False)\n    device = torch.device(device)\n    base = official_codi_base_model(model)\n    body, embedding = base.transformer, model.input_embeddings()\n    model.eval()\n    head.eval()\n    vocabulary_stop = int(model.eot_id)\n    eos = int(tokenizer.eos_token_id)\n    total = sum(len(b.indices) for b in batches)\n    outputs, texts, counts_out = [None]*total, [None]*total, [0]*total\n    with timeline.span(\'answer_cue_tokenization\', gpu=False):\n        cue_ids = tokenizer(\' The answer is:\', add_special_tokens=False)[\'input_ids\'] if mode == \'codi\' else []\n    for batch_number, batch in enumerate(batches):\n        with timeline.metadata(batch_index=batch_number, question_indices=list(batch.indices), token_position=-1):\n            with timeline.span(\'host_to_device_input_ids\'):\n                ids = batch.ids.to(device, non_blocking=True)\n            with timeline.span(\'host_to_device_attention_mask\'):\n                mask = batch.mask.to(device, non_blocking=True)\n            with timeline.span(\'prompt_tensor_construction\'):\n                if mode == \'codi\':\n                    bot = torch.full((len(ids),1), model.bot_id, device=device, dtype=torch.long)\n                    ids = torch.cat((ids,bot),1)\n                    mask = torch.cat((mask,torch.ones_like(bot)),1)\n            # Do not silently truncate GPT-2 context.\n            maximum = getattr(model.config, \'n_positions\', 1024) if hasattr(model, \'config\') else 1024\n            reserve = latent_iterations + 1 + len(cue_ids) if mode == \'codi\' else 0\n            if ids.shape[1] + reserve + max_new_tokens > maximum:\n                raise ValueError(\'Prompt plus generation exceeds context; reduce the configured token cap\')\n            with timeline.span(\'prefill_position_ids\'):\n                positions = mask.long().cumsum(-1)-1 if mode == \'explicit_cot\' else None\n                if positions is not None:\n                    positions.masked_fill_(mask == 0, 1)\n            with timeline.metadata(phase=\'prefill\'):\n                with timeline.span(\'transformer_prefill\'):\n                    prefill_kwargs = dict(input_ids=ids, attention_mask=mask, use_cache=True, return_dict=True)\n                    if getattr(getattr(body, \'config\', None), \'model_type\', None) == \'gpt2\':\n                        from transformers import DynamicCache\n                        prefill_kwargs[\'past_key_values\'] = DynamicCache()\n                    if positions is not None:\n                        prefill_kwargs[\'position_ids\'] = positions\n                    out = body(**prefill_kwargs)\n            with timeline.span(\'kv_cache_reference_update\'):\n                cache = out.past_key_values\n                hidden = out.last_hidden_state[:, -1:, :]\n            if mode == \'codi\':\n                with timeline.metadata(phase=\'latent\'), timeline.span(\'phase_latent\'):\n                    with timeline.metadata(phase=\'latent_projection_initial\'):\n                        with timeline.span(\'latent_projector\'):\n                            latent = model.prj(hidden)\n                    for step in range(latent_iterations):\n                        with timeline.metadata(phase=\'latent\', latent_step=step):\n                            with timeline.span(\'transformer_latent_pass\'):\n                                out = body(inputs_embeds=latent, past_key_values=cache, use_cache=True, return_dict=True)\n                            with timeline.span(\'kv_cache_reference_update\'):\n                                cache = out.past_key_values\n                            with timeline.span(\'latent_projector\'):\n                                latent = model.prj(out.last_hidden_state[:, -1:, :])\n            with timeline.metadata(phase=\'visible_decode\'), timeline.span(\'phase_visible\'):\n                if mode == \'codi\':\n                    with timeline.metadata(phase=\'answer_cue\'):\n                        with timeline.span(\'answer_cue_tensor_construction\'):\n                            cue = torch.tensor([model.eot_id,*cue_ids], device=device).unsqueeze(0).expand(len(ids),-1)\n                        with timeline.span(\'answer_cue_embedding\'):\n                            cue_embedding = embedding(cue)\n                        with timeline.span(\'transformer_answer_cue\'):\n                            out = body(inputs_embeds=cue_embedding, past_key_values=cache, use_cache=True, return_dict=True)\n                        cache = out.past_key_values\n                        hidden = out.last_hidden_state[:, -1:, :]\n                with timeline.span(\'generation_buffer_allocation\'):\n                    tokens = torch.full((len(ids), max_new_tokens), eos, device=device, dtype=torch.long)\n                    counts = torch.zeros(len(ids), device=device, dtype=torch.long)\n                    finished = torch.zeros(len(ids), device=device, dtype=torch.bool)\n                replay = None\n                if forced_tokens is not None:\n                    with timeline.span(\'replay_cpu_padding\', gpu=False):\n                        seqs = [forced_tokens[i] for i in batch.indices]\n                        limit = max(map(len,seqs))\n                        if limit > max_new_tokens or any(not seq for seq in seqs):\n                            raise ValueError(\'Replay tokens must fit the generation cap\')\n                        replay_cpu = torch.full((len(ids),limit), eos, dtype=torch.long)\n                        for row, seq in enumerate(seqs):\n                            replay_cpu[row,:len(seq)] = torch.tensor(seq)\n                    with timeline.span(\'host_to_device_replay_tokens\'):\n                        replay = replay_cpu.to(device)\n                else:\n                    limit = max_new_tokens\n                for position in range(limit):\n                    with timeline.metadata(phase=\'visible_decode\', token_position=position):\n                        if observer:\n                            with timeline.span(\'collect_hidden_state_to_cpu\'):\n                                observer(hidden[:, -1, :], ~finished, position, batch.indices)\n                        with timeline.span(\'lm_head_and_argmax\'):\n                            predicted = select_token(head, hidden[:, -1, :], vocabulary_stop)\n                        with timeline.span(\'token_selection_and_buffers\'):\n                            token = predicted if replay is None else replay[:, position]\n                            active = ~finished\n                            tokens[:,position] = torch.where(active,token,tokens[:,position])\n                            counts += active.long()\n                            finished |= active & (token == eos)\n                        with timeline.span(\'termination_check_host_sync\'):\n                            stop = position+1 == limit or bool(finished.all())\n                        if stop:\n                            break\n                        with timeline.span(\'next_token_embedding\'):\n                            embedded = embedding(token).unsqueeze(1)\n                        with timeline.span(\'decode_attention_mask_update\'):\n                            if mode == \'explicit_cot\':\n                                mask = torch.cat((mask,torch.ones((len(ids),1),device=device,dtype=mask.dtype)),1)\n                        with timeline.span(\'transformer_visible_token\'):\n                            kwargs = dict(inputs_embeds=embedded, past_key_values=cache, use_cache=True, return_dict=True)\n                            if mode == \'explicit_cot\':\n                                kwargs[\'attention_mask\'] = mask\n                                kwargs[\'position_ids\'] = (mask.long().sum(-1)-1).unsqueeze(1)\n                            out = body(**kwargs)\n                        with timeline.span(\'kv_cache_reference_update\'):\n                            cache, hidden = out.past_key_values, out.last_hidden_state\n                with timeline.span(\'device_to_host_token_buffer\'):\n                    cpu_tokens = tokens.cpu()\n                with timeline.span(\'device_to_host_counts\'):\n                    cpu_counts = counts.cpu().tolist()\n                with timeline.span(\'cpu_token_conversion_and_text_decode\', gpu=False):\n                    for row,index in enumerate(batch.indices):\n                        count = int(cpu_counts[row])\n                        seq = tuple(int(t) for t in cpu_tokens[row,:count].tolist())\n                        outputs[index], counts_out[index] = seq, count\n                        texts[index] = tokenizer.decode(seq, skip_special_tokens=True)\n            if not timeline.unified_clock:\n                timeline.resolve()\n    return FastCODIGeneration(tuple(texts),tuple(outputs),tuple(counts_out))\n\n\nclass DenseSelector(nn.Module):\n    def __init__(self, head, vocabulary_size):\n        super().__init__()\n        self.head = head\n        self.vocabulary_size = vocabulary_size\n    def forward(self, hidden):\n        return self.head(hidden)[..., :self.vocabulary_size]\n    def select_token(self, hidden, *, vocabulary_stop):\n        return self(hidden).argmax(-1)\n\n\nclass FixedRankHead(nn.Module):\n    def __init__(self, source, rank=96):\n        super().__init__()\n        self.vocabulary_size = source.vocabulary_size\n        self.down = nn.Linear(source.hidden_size,rank)\n        self.up = nn.Linear(rank,source.vocabulary_size)\n        with torch.no_grad():\n            self.down.weight.copy_(source.down.weight[:rank])\n            self.down.bias.copy_(source.down.bias[:rank])\n            self.up.weight.copy_(source.up.weight[:,:rank])\n            self.up.bias.copy_(source.up.bias)\n        self.requires_grad_(False)\n    def forward(self, hidden):\n        return self.up(self.down(hidden))\n    def select_token(self, hidden, *, vocabulary_stop):\n        return self(hidden).argmax(-1)\n\n\n\nCOLUMNS = (\'Explicit\', \'CODI overall\', \'CODI latent only\', \'CODI visible only\')\nBREAKDOWN_ROWS = (\n    \'Question loading / tokenization\', \'CPU padding / allocation\',\n    \'Input transfer to GPU\', \'Embeddings\',\n    *(f\'Transformer block {i + 1:02d}\' for i in range(12)),\n    \'Transformer final norm\', \'Transformer masks / bookkeeping\',\n    \'Latent projector\', \'LM head\', \'Argmax / head dispatch\',\n    \'Token buffers / cache updates\', \'EOS check / synchronization\',\n    \'Output transfer to CPU\', \'Text decoding\', \'Python / tracing gaps\',\n)\n\n\ndef _category(row, ancestors):\n    # Descendant events inherit their enclosing component. Summing exclusive\n    # durations reconstructs that component without counting nested modules twice.\n    for event in [row, *ancestors]:\n        name = event[\'name\']\n        if name.startswith(\'transformer.h.\'):\n            return f"Transformer block {int(name.split(\'.\')[2]) + 1:02d}"\n        if name.startswith(\'transformer.ln_f\'):\n            return \'Transformer final norm\'\n        if name.startswith((\'transformer.wte\', \'transformer.wpe\')):\n            return \'Embeddings\'\n        if name.startswith(\'projector\'):\n            return \'Latent projector\'\n        if name.startswith(\'lm_head\') and name != \'lm_head_and_argmax\':\n            return \'LM head\'\n    name = row[\'name\']\n    if name in (\'load_question\', \'question_normalization\', \'question_tokenization\', \'answer_cue_tokenization\'):\n        return \'Question loading / tokenization\'\n    if name == \'cpu_padding_and_tensor_allocation\':\n        return \'CPU padding / allocation\'\n    if name.startswith(\'host_to_device\'):\n        return \'Input transfer to GPU\'\n    if name in (\'next_token_embedding\', \'answer_cue_embedding\'):\n        return \'Embeddings\'\n    if name.startswith(\'transformer_\') or name in (\'prefill_position_ids\', \'decode_attention_mask_update\'):\n        return \'Transformer masks / bookkeeping\'\n    if name == \'latent_projector\':\n        return \'Latent projector\'\n    if name == \'lm_head_and_argmax\':\n        return \'Argmax / head dispatch\'\n    if name == \'termination_check_host_sync\':\n        return \'EOS check / synchronization\'\n    if name.startswith(\'device_to_host\'):\n        return \'Output transfer to CPU\'\n    if name == \'cpu_token_conversion_and_text_decode\':\n        return \'Text decoding\'\n    if name in (\'kv_cache_reference_update\', \'generation_buffer_allocation\',\n                \'token_selection_and_buffers\', \'prompt_tensor_construction\', \'answer_cue_tensor_construction\'):\n        return \'Token buffers / cache updates\'\n    return \'Python / tracing gaps\'\n\n\ndef partition_question(events):\n    """Exclusive intervals on one clock; never add CPU and CUDA measurements."""\n    by_id = {r[\'event_id\']: r for r in events}\n    roots = [r for r in events if r[\'name\'] == \'question_total\']\n    if len(roots) != 1:\n        raise ValueError(\'Expected exactly one complete question trace\')\n    root = roots[0]\n    clock = \'cuda_stream_ms\' if root.get(\'cuda_stream_ms\') is not None else \'cpu_wall_ms\'\n    if any(r.get(clock) is None for r in events):\n        raise ValueError(\'Every span must use the same clock; enable unified_clock\')\n    children = {}\n    for row in events:\n        children.setdefault(row.get(\'parent_id\'), []).append(row)\n    values = {}\n    for row in events:\n        exclusive = row[clock] - sum(c[clock] for c in children.get(row[\'event_id\'], []))\n        if exclusive < -0.01:\n            raise ValueError(f"Overlapping timing spans: {row[\'name\']}: {exclusive} ms")\n        # Keep sub-microsecond event rounding differences so totals reconcile.\n        ancestors = []\n        parent = row.get(\'parent_id\')\n        while parent is not None:\n            ancestors.append(by_id[parent]); parent = by_id[parent].get(\'parent_id\')\n        category = _category(row, ancestors)\n        phase = row.get(\'phase\', \'shared\')\n        phase = (\'latent\' if phase in (\'latent\', \'latent_projection_initial\') else\n                 \'visible\' if phase in (\'visible_decode\', \'answer_cue\') else \'shared\')\n        key = (category, phase)\n        values[key] = values.get(key, 0.0) + exclusive\n    return dict(mode=root[\'mode\'], question_id=root.get(\'question_id\'), repeat=root.get(\'repeat\'),\n                total_ms=root[clock], clock=clock,\n                breakdown=[dict(name=name, phase=phase, ms=value) for (name, phase), value in values.items()])\n\n\ndef bottleneck_means(samples, *, per_token=False):\n    """Question means or aggregate time / native generated-step count.\n\n    Token units: explicit = visible output token; CODI overall = latent + visible\n    step; CODI latent = latent step; CODI visible = visible output token. Prompt and\n    forced cue work are amortized, not added to the generated-step denominator.\n    """\n    groups = {mode: [s for s in samples if s[\'mode\'] == mode] for mode in (\'explicit_cot\', \'codi\')}\n    if any(not group for group in groups.values()):\n        raise ValueError(\'Both reasoning modes need timing samples\')\n    rows = {name: dict.fromkeys(COLUMNS, 0.0) for name in BREAKDOWN_ROWS}\n    totals = dict.fromkeys(COLUMNS, 0.0)\n    for mode, group in groups.items():\n        overall = \'Explicit\' if mode == \'explicit_cot\' else \'CODI overall\'\n        if per_token:\n            visible = sum(s[\'visible_tokens\'] for s in group)\n            latent = sum(s[\'latent_steps\'] for s in group)\n            denominators = {overall: visible + latent if mode == \'codi\' else visible,\n                            \'CODI latent only\': latent, \'CODI visible only\': visible}\n            if denominators[overall] <= 0 or (mode == \'codi\' and min(latent, visible) <= 0):\n                raise ValueError(\'Per-token reporting needs positive generated-step counts\')\n        else:\n            denominators = dict.fromkeys(COLUMNS, len(group))\n        for sample in group:\n            totals[overall] += sample[\'total_ms\'] / denominators[overall]\n            for item in sample[\'breakdown\']:\n                value = item[\'ms\'] / denominators[overall]\n                rows[item[\'name\']][overall] += value\n                if mode == \'codi\' and item[\'phase\'] in (\'latent\', \'visible\'):\n                    column = \'CODI latent only\' if item[\'phase\'] == \'latent\' else \'CODI visible only\'\n                    phase_value = item[\'ms\'] / denominators[column]\n                    rows[item[\'name\']][column] += phase_value\n                    totals[column] += phase_value\n    label = \'Total average time per token/step\' if per_token else \'Total average time\'\n    return [(label, totals), *rows.items(), (label + \' (repeat)\', dict(totals))]\n\n\n@torch.inference_mode()\ndef profile_question(model, tokenizer, head, question, *, mode, device, max_new_tokens,\n                     question_id=0, repeat=0, arm=\'rank96\', latent_iterations=6):\n    """One batch-1 sample; only displayed components receive module hooks."""\n    device = torch.device(device)\n    timeline = Timeline(device, unified_clock=True)\n    timeline.context.update(mode=mode, arm=arm, question_id=question_id, repeat=repeat, phase=\'shared\')\n    base = official_codi_base_model(model)\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n    roots = [(f\'transformer.h.{i}\', block) for i, block in enumerate(base.transformer.h)]\n    roots += [(\'transformer.\' + name, getattr(base.transformer, name))\n              for name in (\'wte\', \'wpe\', \'ln_f\') if hasattr(base.transformer, name)]\n    roots += [(\'projector\', model.prj), (\'lm_head\', head)]\n    with timeline.modules(roots, recursive=False):\n        with timeline.span(\'question_total\'):\n            with timeline.span(\'load_question\', gpu=False):\n                questions = [str(question)]\n            batches = prepare_questions(tokenizer, questions, 1, timeline)\n            result = decode(model, tokenizer, batches, head, mode=mode, device=device,\n                            max_new_tokens=max_new_tokens, latent_iterations=latent_iterations, timeline=timeline)\n    timeline.resolve()\n    sample = partition_question(timeline.records)\n    sample.update(arm=arm, tokens=list(result.token_ids[0]), text=result.texts[0],\n                  prompt_tokens=int(batches[0].mask.sum()),\n                  latent_steps=latent_iterations if mode == \'codi\' else 0,\n                  visible_tokens=result.generated_token_counts[0])\n    return sample, timeline.records\n'
EXPERIMENT_SOURCE_SHA256 = 'b03d971e9ee407bcbb5f683a14f586d5c8632d8520bdd249ad48d3bc15bfb1a0'
runtime_path=pathlib.Path('/kaggle/working/dual_global_head_runtime.py')
runtime_path.write_text(RUNTIME_SOURCE)
spec=importlib.util.spec_from_file_location('dual_global_head_runtime',runtime_path)
runtime=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=runtime
spec.loader.exec_module(runtime)
import torch
import pandas as pd
from dataclasses import asdict
from importlib.metadata import version
from src.mech.global_low_rank_head import (
    NestedLowRankVocabularyHead,activation_whitened_factors,distil_nested_head,evaluate_nested_head)
from src.models.official_codi import (
    build_official_codi_gpt2,download_official_checkpoint,load_official_checkpoint,official_codi_base_model)
from src.inference.official_codi_fast import (
    generate_official_codi_fast,prepare_official_codi_batches,merge_official_codi_lora_)
from src.utils.config import load_config
from src.data.answer_extract import answers_match,normalize_gold
assert torch.cuda.is_available(),'Select Settings > Accelerator > GPU T4 x2.'
assert 'T4' in torch.cuda.get_device_name(0),'Select GPU T4 x2 in Kaggle settings and restart the session.'
device=torch.device('cuda:0')
DEPLOY_DTYPE=torch.float16
torch.manual_seed(SEED); random.seed(SEED)
config=dict(seed=SEED,fit=FIT_QUESTIONS,selection=SELECT_QUESTIONS,recovery=RECOVERY_QUESTIONS,
    state_caps=[MAX_FIT_STATES,MAX_SELECT_STATES,MAX_RECOVERY_STATES],ranks=RANKS,
    epochs=[CLEAN_EPOCHS,RECOVERY_EPOCHS],token_caps=MAX_NEW_TOKENS,
    questions=TIMING_QUESTIONS,repeats=TIMING_REPEATS,batch_size=1,heads=['dense','rank96_eager'],accuracy='full_eligible_set_for_each_dataset', models=['gpt2','smollm2_135m','qwen2_5_0_5b'], datasets=['gsm8k','svamp','asdiv'],
    base=BASE_COMMIT,source=EXPERIMENT_SOURCE_SHA256,torch=torch.__version__,cuda=torch.version.cuda,
    packages={name:version(name) for name in ('transformers','peft','huggingface_hub','hf_xet','accelerate')},
    gpu=torch.cuda.get_device_name(0))
RUN_DIR=OUTPUT_ROOT/hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR.mkdir(parents=True,exist_ok=True)
DEBUG_PATH=RUN_DIR/'debug.jsonl.gz'
# Preserve raw measurements/predictions when completed model/dataset pairs are reused.
if not DEBUG_PATH.exists():
    with gzip.open(DEBUG_PATH,'wt') as stream: pass

def debug_record(kind,payload):
    with gzip.open(DEBUG_PATH,'at',encoding='utf-8') as stream:
        stream.write(json.dumps(dict(kind=kind,data=payload),default=str)+'\n')
def save_json(path,value):
    temp=pathlib.Path(str(path)+'.tmp'); temp.write_text(json.dumps(value,indent=2,default=str)); temp.replace(path)
def save_pt(path,value):
    temp=pathlib.Path(str(path)+'.tmp'); torch.save(value,temp); temp.replace(path)
setup=runtime.Timeline(device)
setup.context.update(mode='setup')
setup.records.extend(BOOTSTRAP_TIMES)
def flush_setup():
    setup.resolve()
    for row in setup.records: debug_record('setup',row)
    setup.records.clear()
debug_record('manifest',config)
debug_record('dependency_log',setup_log.read_text())
flush_setup()
# These two construction notices are expected in the pinned official loader.
# Preserve them in the debug log and keep unrelated warnings visible.
class KnownConstructionNotice(logging.Filter):
    def filter(self,record):
        if record.getMessage().startswith('The new embeddings will be initialized'):
            debug_record('construction_notice',record.getMessage()); return False
        return True
logging.getLogger('transformers.modeling_utils').addFilter(KnownConstructionNotice())
print(f"Using {config['gpu']}; 3 models x 3 datasets; rank 96, FP16, batch 1.")

In [ ]:
ABLATION_SOURCE = '"""Model/dataset adapters for the fixed global-head experiment.\n\nNo compiler, quantization, batch-size or deployment-rank sweep lives here.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport random\nimport re\nimport time\nimport xml.etree.ElementTree as ET\nfrom pathlib import Path\nfrom urllib.request import urlopen\nfrom urllib.error import HTTPError, URLError\n\nimport torch\nfrom torch import nn\n\ntry:\n    import dual_global_head_runtime as reference\nexcept ModuleNotFoundError:\n    from src.inference import global_head_comparison as reference\nfrom src.data.answer_extract import normalize_gold, normalize_number\nfrom src.models.official_codi import official_codi_base_model\n\n\nMODEL_SPECS = (\n    dict(key=\'gpt2\', label=\'GPT-2 / released CODI\', repo=\'zen-E/CODI-gpt2\'),\n    dict(key=\'smollm2_135m\', label=\'SmolLM2-135M\', repo=\'HuggingFaceTB/SmolLM2-135M\',\n         revision=\'93efa2f097d58c2a74874c7e644dbc9b0cee75a2\'),\n    dict(key=\'qwen2_5_0_5b\', label=\'Qwen2.5-0.5B\', repo=\'Qwen/Qwen2.5-0.5B\',\n         revision=\'060db6499f32faf8b98477b0a26969ef7d8b9987\'),\n)\nDATASETS = (\'gsm8k\', \'svamp\', \'asdiv\')\nGSM_REVISION = \'3101c7d5072418e28b9008a6636bde82a006892c\'\nSVAMP_URL = \'https://raw.githubusercontent.com/arkilpatel/SVAMP/78e727689e1c1bebfc4be39c446898e8e10b0518/SVAMP.json\'\nASDIV_URL = \'https://raw.githubusercontent.com/chaochun/nlu-asdiv-dataset/883f90a9a65bf00304ba8f37423910fe743abc47/dataset/ASDiv.xml\'\nDATA_HASHES = {\n    \'svamp\': \'5be77703a6d891ae476d7c082787ad361392aa02453b132516cdd5f4e7934e3e\',\n    \'asdiv\': \'ef8904068482919ac48c8eeaaf6df344b8a308ba66d048c2d4d87eab82dc4929\',\n}\nSCALAR_ANSWER = re.compile(r\'([-+]?\\$?(?:\\d[\\d,]*(?:\\.\\d*)?|\\.\\d+)(?:%)?)(?:\\s*\\([^)]*\\))?\\s*\')\n\n\ndef question_key(question):\n    return \' \'.join(str(question).casefold().split())\n\n\ndef parse_svamp(payload):\n    rows = []\n    for row in json.loads(payload):\n        gold = normalize_number(str(row[\'Answer\']))\n        if gold is None:\n            raise ValueError(f"Invalid SVAMP answer: {row[\'ID\']}")\n        rows.append(dict(id=row[\'ID\'], question=f"{row[\'Body\'].strip()} {row[\'Question\'].strip()}", gold=str(gold)))\n    return rows\n\n\ndef parse_asdiv(payload):\n    """Keep only single scalar gold answers compatible with the existing scorer.\n\n    Do not turn \'3:30\' into 30, \'5; 15; 20\' into 20, or \'February 3rd\' into 3.\n    Units in parentheses are stripped before numeric normalization.\n    """\n    rows, excluded = [], []\n    for problem in ET.fromstring(payload).findall(\'.//Problem\'):\n        answer = problem.findtext(\'Answer\', \'\').strip()\n        match = SCALAR_ANSWER.fullmatch(answer)\n        if match is None:\n            excluded.append(dict(id=problem.get(\'ID\'), answer=answer, reason=\'not a single scalar numeric answer\'))\n            continue\n        gold = normalize_number(match.group(1))\n        if gold is None:\n            raise ValueError(f\'Invalid ASDiv scalar answer: {answer}\')\n        question = \' \'.join(problem.findtext(name, \'\').strip() for name in (\'Body\', \'Question\'))\n        rows.append(dict(id=problem.get(\'ID\'), question=question, gold=str(gold)))\n    return rows, excluded\n\n\ndef download_datasets(cache_dir):\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    sources = {}\n    def read(key, url):\n        path = cache_dir / key\n        if not path.exists():\n            for attempt in range(3):\n                try:\n                    with urlopen(url, timeout=60) as response:\n                        payload = response.read()\n                    break\n                except (HTTPError, URLError, TimeoutError):\n                    if attempt == 2: raise\n                    time.sleep(2*(attempt+1))\n            if key in DATA_HASHES and hashlib.sha256(payload).hexdigest() != DATA_HASHES[key]:\n                raise ValueError(f\'{key}: downloaded dataset checksum mismatch\')\n            temp = path.with_suffix(\'.tmp\')\n            temp.write_bytes(payload)\n            temp.replace(path)\n        payload = path.read_bytes()\n        sha = hashlib.sha256(payload).hexdigest()\n        if key in DATA_HASHES and sha != DATA_HASHES[key]:\n            raise ValueError(f\'{key}: dataset checksum mismatch; remove the corrupt cached file and retry\')\n        sources[key] = dict(url=url, sha256=sha)\n        return payload\n    prefix = f\'https://raw.githubusercontent.com/openai/grade-school-math/{GSM_REVISION}/grade_school_math/data/\'\n    train = [json.loads(line) for line in read(\'gsm8k_train\', prefix+\'train.jsonl\').splitlines() if line.strip()]\n    test = [json.loads(line) for line in read(\'gsm8k_test\', prefix+\'test.jsonl\').splitlines() if line.strip()]\n    evaluation = {\'gsm8k\': [dict(id=f\'gsm8k-test-{i}\', question=r[\'question\'],\n                         gold=str(normalize_gold(r[\'answer\'], \'gsm8k_main\'))) for i, r in enumerate(test)]}\n    evaluation[\'svamp\'] = parse_svamp(read(\'svamp\', SVAMP_URL))\n    evaluation[\'asdiv\'], excluded = parse_asdiv(read(\'asdiv\', ASDIV_URL))\n    if (len(train), len(test), len(evaluation[\'svamp\']), len(evaluation[\'asdiv\']), len(excluded)) != (7473,1319,1000,2084,221):\n        raise ValueError(\'Pinned dataset counts changed\')\n    if any(r[\'gold\']==\'None\' for rows in evaluation.values() for r in rows):\n        raise ValueError(\'Missing numeric gold\')\n    return train, evaluation, dict(sources=sources, asdiv_excluded=excluded)\n\n\ndef partition_train(train, evaluation, seed=89):\n    """The same GSM8K fit/selection/recovery/timing/warmup split as before."""\n    unique = {}\n    for row in train:\n        unique.setdefault(question_key(row[\'question\']), dict(question=str(row[\'question\']), gold=str(row[\'answer\'])))\n    rows = list(unique.values())\n    random.Random(seed).shuffle(rows)\n    sizes = dict(fit=1024, selection=256, recovery=256, timing=16, warmup=4)\n    splits, start = {}, 0\n    for name, size in sizes.items():\n        splits[name] = rows[start:start+size]\n        start += size\n    if start > len(rows):\n        raise ValueError(\'Insufficient unique GSM8K training questions\')\n    used = {question_key(r[\'question\']) for name in (\'fit\',\'selection\',\'recovery\',\'warmup\') for r in splits[name]}\n    for dataset, questions in evaluation.items():\n        overlap = used & {question_key(r[\'question\']) for r in questions}\n        if overlap:\n            raise ValueError(f\'{dataset}: {len(overlap)} evaluation questions overlap fitting/warmup data\')\n    return splits\n\n\ndef timing_questions(dataset, splits, evaluation, seed=89):\n    if dataset == \'gsm8k\':\n        return splits[\'timing\']\n    rows = list(evaluation[dataset])\n    random.Random(seed).shuffle(rows)\n    return rows[:16]\n\n\nclass NoLatentProjector(nn.Module):\n    def forward(self, x):\n        raise RuntimeError(\'This backbone has no trained CODI projector\')\n\n\nclass BackboneView(nn.Module):\n    """Expose the original body/head under the existing decoder\'s access contract."""\n    def __init__(self, backbone):\n        super().__init__()\n        self.backbone = backbone\n        self.config = backbone.config\n        self.config.n_positions = int(self.config.max_position_embeddings)\n\n    @property\n    def transformer(self):\n        return self.backbone.model\n\n    def get_output_embeddings(self):\n        return self.backbone.get_output_embeddings()\n\n    def generate(self, *args, **kwargs):\n        return self.backbone.generate(*args, **kwargs)\n\n\nclass ExplicitBackbone(nn.Module):\n    """Original pretrained model, without fabricated latent-reasoning weights."""\n    def __init__(self, backbone):\n        super().__init__()\n        self.codi = BackboneView(backbone)\n        self.eot_id = int(backbone.get_output_embeddings().weight.shape[0])\n        self.prj = NoLatentProjector()\n        self.has_codi = False\n\n    @property\n    def config(self):\n        return self.codi.config\n\n    def input_embeddings(self):\n        return self.codi.backbone.get_input_embeddings()\n\n\ndef decode(model, tokenizer, batches, head, *, mode, **kwargs):\n    if mode == \'codi\' and getattr(model, \'has_codi\', True) is False:\n        raise ValueError(\'CODI is unavailable for this untrained backbone\')\n    return reference.decode(model, tokenizer, batches, head, mode=mode, **kwargs)\n\n\ndef block_roots(model):\n    body = official_codi_base_model(model).transformer\n    blocks = body.h if hasattr(body, \'h\') else body.layers\n    roots = [(f\'transformer.h.{i}\', block) for i, block in enumerate(blocks)]\n    # Use the existing categories without renaming or changing actual model modules.\n    if hasattr(body, \'wte\'):\n        roots += [(\'transformer.\'+name, getattr(body,name)) for name in (\'wte\',\'wpe\',\'ln_f\')]\n    else:\n        roots += [(\'transformer.wte\', body.embed_tokens), (\'transformer.ln_f\', body.norm)]\n    return roots, len(blocks)\n\n\n@torch.inference_mode()\ndef profile_question(model, tokenizer, head, question, *, mode, device, max_new_tokens,\n                     question_id=0, repeat=0, arm=\'rank96\', latent_iterations=6):\n    trace = reference.Timeline(device, unified_clock=True)\n    trace.context.update(mode=mode, arm=arm, question_id=question_id, repeat=repeat, phase=\'shared\')\n    roots, _ = block_roots(model)\n    roots += [(\'projector\', model.prj), (\'lm_head\', head)]\n    if torch.device(device).type == \'cuda\':\n        torch.cuda.synchronize(device)\n    with trace.modules(roots, recursive=False):\n        with trace.span(\'question_total\'):\n            with trace.span(\'load_question\', gpu=False):\n                questions = [str(question)]\n            batches = reference.prepare_questions(tokenizer, questions, 1, trace)\n            result = decode(model, tokenizer, batches, head, mode=mode, device=device,\n                max_new_tokens=max_new_tokens, latent_iterations=latent_iterations, timeline=trace)\n    trace.resolve()\n    sample = reference.partition_question(trace.records)\n    sample.update(arm=arm, tokens=list(result.token_ids[0]), text=result.texts[0],\n        prompt_tokens=int(batches[0].mask.sum()), latent_steps=latent_iterations if mode==\'codi\' else 0,\n        visible_tokens=result.generated_token_counts[0])\n    return sample, trace.records\n\n\ndef bottleneck_means(samples, *, n_layers, per_token=False):\n    """Keep four columns; absent CODI is NaN, not a misleading zero."""\n    names = list(reference.BREAKDOWN_ROWS)\n    first = names.index(\'Transformer block 01\')\n    names = names[:first] + [f\'Transformer block {i+1:02d}\' for i in range(n_layers)] + names[first+12:]\n    columns = reference.COLUMNS\n    rows = {name: dict.fromkeys(columns, float(\'nan\')) for name in names}\n    totals = dict.fromkeys(columns, float(\'nan\'))\n    for mode in (\'explicit_cot\',\'codi\'):\n        group = [s for s in samples if s[\'mode\']==mode]\n        if not group:\n            continue\n        active = [\'Explicit\'] if mode==\'explicit_cot\' else list(columns[1:])\n        for column in active:\n            totals[column] = 0.\n            for row in rows.values(): row[column] = 0.\n        overall = active[0]\n        visible = sum(s[\'visible_tokens\'] for s in group)\n        latent = sum(s[\'latent_steps\'] for s in group)\n        den = dict.fromkeys(active, len(group))\n        if per_token:\n            den[overall] = visible + (latent if mode==\'codi\' else 0)\n            if mode==\'codi\': den.update({\'CODI latent only\':latent, \'CODI visible only\':visible})\n        if min(den.values()) <= 0: raise ValueError(\'Positive generation counts required\')\n        for sample in group:\n            totals[overall] += sample[\'total_ms\']/den[overall]\n            for part in sample[\'breakdown\']:\n                rows[part[\'name\']][overall] += part[\'ms\']/den[overall]\n                if mode==\'codi\' and part[\'phase\'] in (\'latent\',\'visible\'):\n                    col = \'CODI \'+part[\'phase\']+\' only\'\n                    rows[part[\'name\']][col] += part[\'ms\']/den[col]\n                    totals[col] += part[\'ms\']/den[col]\n    if all(torch.isnan(torch.tensor(x)) for x in totals.values()): raise ValueError(\'No samples\')\n    return [(\'Total average time\',totals), *rows.items(), (\'Total average time (repeat)\',dict(totals))]\n'
adapter_path=pathlib.Path('/kaggle/working/model_dataset_ablation_runtime.py')
adapter_path.write_text(ABLATION_SOURCE)
spec=importlib.util.spec_from_file_location('model_dataset_ablation_runtime',adapter_path)
ablation=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=ablation
spec.loader.exec_module(ablation)
from types import SimpleNamespace
from functools import partial
from IPython.display import HTML,display
import html
from transformers import AutoModelForCausalLM,AutoTokenizer
REFERENCE_RUNTIME=runtime
EXPERIMENT_DIR=RUN_DIR
MODEL_SPECS=ablation.MODEL_SPECS
all_results={}

train,evaluation,data_manifest=ablation.download_datasets(EXPERIMENT_DIR/'datasets')
base_splits=ablation.partition_train(train,evaluation,SEED)
debug_record('datasets',data_manifest)
debug_record('partitions',base_splits)
for name,rows in evaluation.items(): debug_record('evaluation_questions',dict(dataset=name,rows=rows))
print('Accuracy sets: GSM8K 1,319; SVAMP 1,000; ASDiv numeric subset 2,084.')
print('ASDiv excludes 221 non-scalar answers (times, ratios, names, multiple answers); exclusions are logged.')
print('SVAMP/ASDiv are evaluation-only: no head refitting or selection on them.')

In [ ]:
GPT2_LOAD = "with setup.span('config_load',gpu=False): cfg=load_config('configs/official_codi_gpt2.yaml')\nwith setup.span('checkpoint_download_and_hash',gpu=False):\n    checkpoint=download_official_checkpoint(repo_id=cfg.checkpoint.repo_id,revision=cfg.checkpoint.revision,\n        filename=cfg.checkpoint.filename,expected_sha256=cfg.checkpoint.sha256)\nwith setup.span('gpt2_model_and_tokenizer_load',gpu=False):\n    with warnings.catch_warnings(record=True) as notices:\n        warnings.filterwarnings('always',message='fan_in_fan_out is set to False.*')\n        model,tokenizer=build_official_codi_gpt2(base_model=cfg.model.base_model,base_revision=cfg.model.base_revision,\n            dtype=torch.float32,settings=cfg.model)\n    for notice in notices:\n        if 'fan_in_fan_out is set to False' in str(notice.message):\n            debug_record('construction_notice',str(notice.message))\n        else: warnings.warn(str(notice.message),notice.category)\nwith setup.span('checkpoint_load_and_verification',gpu=False):\n    load_report=load_official_checkpoint(model,checkpoint,expected_sha256=cfg.checkpoint.sha256)\n    debug_record('checkpoint',asdict(load_report))\nwith setup.span('model_host_to_device_fp32'): model.requires_grad_(False).to(device).eval()\nbase=official_codi_base_model(model)\nfull_head=base.get_output_embeddings()\nweight=full_head.weight[:model.eot_id].detach()\nbias=None if getattr(full_head,'bias',None) is None else full_head.bias[:model.eot_id].detach()\nflush_setup()"
PARITY_CHECK = "parity_questions=[r['question'] for r in splits['selection'][:4]]\nparity={}\nwith torch.no_grad():\n    for mode in MODES:\n        prepared=runtime.prepare_questions(tokenizer,parity_questions,4)\n        observed=runtime.decode(model,tokenizer,prepared,full_head,mode=mode,device=device,\n            max_new_tokens=MAX_NEW_TOKENS[mode],latent_iterations=6)\n        if mode=='codi':\n            reference=generate_official_codi_fast(model,tokenizer,\n                prepare_official_codi_batches(tokenizer,parity_questions,batch_size=4,length_bucketed=False),\n                latent_iterations=6,max_new_tokens=MAX_NEW_TOKENS[mode],device=device,answer_cue='The answer is:')\n            expected=reference.token_ids\n        else:\n            from transformers import LogitsProcessor,LogitsProcessorList\n            class VocabularyBoundary(LogitsProcessor):\n                def __call__(self,input_ids,scores):\n                    scores[:,int(model.eot_id):]=float('-inf'); return scores\n            batch=prepared[0]\n            generated=base.generate(input_ids=batch.ids.to(device),attention_mask=batch.mask.to(device),\n                do_sample=False,max_new_tokens=MAX_NEW_TOKENS[mode],pad_token_id=tokenizer.pad_token_id,\n                eos_token_id=tokenizer.eos_token_id,logits_processor=LogitsProcessorList([VocabularyBoundary()]))\n            expected=[]\n            for seq in generated[:,batch.ids.shape[1]:].cpu().tolist():\n                if tokenizer.eos_token_id in seq: seq=seq[:seq.index(tokenizer.eos_token_id)+1]\n                expected.append(tuple(seq))\n            expected=tuple(expected)\n        parity[mode]=dict(examples=len(expected),exact=observed.token_ids==expected)\n        assert parity[mode]['exact'],f'{mode}: custom decoder differs from reference; stop before fitting'\ndebug_record('decoder_parity',parity)\nprint('Decoder checks passed:',MODEL_KEY,MODES)"
FIT_HEADS = "def collect(mode,head,population,cap,tag):\n    path=RUN_DIR/f'states_{mode}_{tag}.pt'\n    if path.exists():\n        with setup.span('state_cache_disk_load',gpu=False,mode=mode,tag=tag):\n            return torch.load(path,map_location='cpu',weights_only=False)\n    chunks=[]; positions=[]\n    def observe(hidden,active,position,indices):\n        chunks.append(hidden[active].detach().cpu().float())\n        positions.extend([position]*int(active.sum()))\n    questions=[r['question'] for r in splits[population]]\n    batches=runtime.prepare_questions(tokenizer,questions,COLLECT_BATCH_SIZE)\n    with setup.span('trajectory_collection',mode=mode,population=population):\n        runtime.decode(model,tokenizer,batches,head,mode=mode,device=device,\n                       max_new_tokens=MAX_NEW_TOKENS[mode],observer=observe)\n    values=torch.cat(chunks)\n    indices=torch.randperm(len(values),generator=torch.Generator().manual_seed(SEED))[:cap]\n    result=dict(states=values[indices].clone(),positions=torch.tensor(positions)[indices],observed_states=len(values))\n    with setup.span('state_cache_disk_save',gpu=False,mode=mode,tag=tag): save_pt(path,result)\n    return result\n\ndef evaluate_positions(head,bundle):\n    result={}\n    for rank in RANKS:\n        result[str(rank)]={}\n        for label,mask in [('all',torch.ones(len(bundle['states']),dtype=torch.bool)),\n                           ('p0',bundle['positions']==0),('p1',bundle['positions']==1),('p2plus',bundle['positions']>=2)]:\n            result[str(rank)][label]=dict(states=int(mask.sum()),**(evaluate_nested_head(\n                head,bundle['states'][mask],weight,readout_bias=bias,rank=rank,batch_size=DISTILL_BATCH_SIZE)\n                if mask.any() else {}))\n    return result\n\nfor mode in MODES:\n    artifact=RUN_DIR/f'global_head_{mode}.pt'\n    if artifact.exists(): print('Reuse fitted head:',mode); continue\n    print('Collect/fitting:',mode,flush=True)\n    fit=collect(mode,full_head,'fit',MAX_FIT_STATES,'fit')\n    selection=collect(mode,full_head,'selection',MAX_SELECT_STATES,'selection')\n    with setup.span('activation_whitened_initialization',mode=mode):\n        centre,down,up,out_bias,init=activation_whitened_factors(fit['states'],weight,96,\n            readout_bias=bias,seed=SEED,compute_device=device)\n        head=NestedLowRankVocabularyHead.from_whitened_factors(centre,down,up,out_bias,RANKS).to(device)\n    initial_metrics=evaluate_positions(head,selection)\n    with setup.span('clean_distillation',mode=mode):\n        clean=distil_nested_head(head,fit['states'],selection['states'],weight,readout_bias=bias,\n            epochs=CLEAN_EPOCHS,batch_size=DISTILL_BATCH_SIZE,learning_rate=2e-4,seed=SEED)\n    head.disable_adaptive(); head.set_rank(64)\n    recovery=collect(mode,head,'recovery',MAX_RECOVERY_STATES,'onpolicy')\n    with setup.span('recovery_distillation',mode=mode):\n        recovered=distil_nested_head(head,torch.cat((fit['states'],recovery['states'])),\n            selection['states'],weight,readout_bias=bias,epochs=RECOVERY_EPOCHS,\n            batch_size=DISTILL_BATCH_SIZE,learning_rate=2e-4,seed=SEED+1)\n    report=dict(mode=mode,initialization=asdict(init),clean=asdict(clean),recovery=asdict(recovered),\n                initial=initial_metrics,final=evaluate_positions(head,selection),\n                fit_states=len(fit['states']),recovery_states=len(recovery['states']))\n    with setup.span('trained_head_disk_save',gpu=False,mode=mode):\n        save_pt(artifact,dict(state_dict={k:v.detach().cpu().clone() for k,v in head.state_dict().items()},report=report))\n    flush_setup()\n    del head,fit,selection,recovery,centre,down,up,out_bias\n    gc.collect(); torch.cuda.empty_cache()\nif MODEL_KEY=='gpt2':\n    with setup.span('lora_merge'): merge_official_codi_lora_(model)\nwith setup.span('model_fp16_conversion'): model.to(dtype=DEPLOY_DTYPE).eval()\nbase=official_codi_base_model(model); full_head=base.get_output_embeddings()\ndel weight,bias\nflush_setup()"
TIMING_RUN = "# Timing runner: only one deployed method, one batch size, and two reasoning modes.\n@torch.inference_mode()\ndef clean_generation(mode,head,question):\n    if device.type=='cuda': torch.cuda.synchronize(device)\n    start=time.perf_counter()\n    batches=runtime.prepare_questions(tokenizer,[question],1)\n    result=runtime.decode(model,tokenizer,batches,head,mode=mode,device=device,max_new_tokens=MAX_NEW_TOKENS[mode])\n    if device.type=='cuda': torch.cuda.synchronize(device)\n    return result,1000*(time.perf_counter()-start)\n\nheads={}\nfor mode in MODES:\n    with setup.span('fitted_head_load_and_move',mode=mode):\n        payload=torch.load(RUN_DIR/f'global_head_{mode}.pt',map_location='cpu',weights_only=False)\n        nested=NestedLowRankVocabularyHead(int(model.config.hidden_size),int(model.eot_id),RANKS)\n        nested.load_state_dict(payload['state_dict'])\n        heads[mode]=runtime.FixedRankHead(nested,96).to(device=device,dtype=DEPLOY_DTYPE).eval()\n        agreement=payload['report']['final']['96']['all']['top1_agreement']\n        print(f'{mode}: fitted head validation token agreement {agreement:.1%}.')\n        debug_record('fit_report',payload['report'])\n    del payload,nested\n\ndense=runtime.DenseSelector(full_head,int(model.eot_id))\nwith setup.span('generation_warmup'):\n    for mode in MODES:\n        for row in splits['warmup']:\n            for head in (dense,heads[mode]): clean_generation(mode,head,row['question'])\n        # Warm both instrumented paths; discard these traces.\n        for arm,head in [('dense',dense),('rank96',heads[mode])]:\n            runtime.profile_question(model,tokenizer,head,splits['warmup'][0]['question'],\n                mode=mode,device=device,max_new_tokens=MAX_NEW_TOKENS[mode],arm=arm)\nflush_setup()\n\nclean_samples=[]\nprofile_samples=[]\nrng=random.Random(SEED)\nfor repeat in range(TIMING_REPEATS):\n    # Finish clean measurements before attaching any profiling hooks.\n    jobs=[(mode,i) for mode in MODES for i in range(len(splits['timing']))]\n    rng.shuffle(jobs)\n    expected={}\n    for mode,i in jobs:\n        arms=[('dense',dense),('rank96',heads[mode])]; rng.shuffle(arms)\n        for arm,head in arms:\n            result,elapsed=clean_generation(mode,head,splits['timing'][i]['question'])\n            row=dict(mode=mode,arm=arm,question_id=i,repeat=repeat,ms=elapsed,\n                     visible_tokens=result.generated_token_counts[0],tokens=list(result.token_ids[0]),text=result.texts[0])\n            clean_samples.append(row)\n            debug_record('clean_sample',row)\n            expected[mode,arm,i]=row['tokens']\n    for mode,i in jobs:\n        arms=[('dense',dense),('rank96',heads[mode])]; rng.shuffle(arms)\n        for arm,head in arms:\n            sample,events=runtime.profile_question(model,tokenizer,head,splits['timing'][i]['question'],\n                mode=mode,device=device,max_new_tokens=MAX_NEW_TOKENS[mode],question_id=i,repeat=repeat,arm=arm)\n            assert sample['tokens']==expected[mode,arm,i],'Instrumentation changed generated tokens'\n            profile_samples.append(sample)\n            debug_record('profile_sample',sample)\n            debug_record('events',events)\n            del events\n    print(f'Timing repeat {repeat+1}/{TIMING_REPEATS} complete.',flush=True)"
ACCURACY_RUN = '# Accuracy is generation accuracy, not agreement with the dense head.\n# Batch 1 matches timing and avoids changing padding/cache behavior for this check.\naccuracy_results={}\nwith torch.inference_mode():\n    for mode in MODES:\n        for arm,head in [(\'dense\',dense),(\'rank96\',heads[mode])]:\n            correct=0; capped=0\n            for i,row in enumerate(test_rows):\n                batches=runtime.prepare_questions(tokenizer,[row[\'question\']],1)\n                result=runtime.decode(model,tokenizer,batches,head,mode=mode,device=device,max_new_tokens=MAX_NEW_TOKENS[mode])\n                matched=answers_match(result.texts[0],row[\'gold\'])\n                hit_cap=result.token_ids[0][-1]!=tokenizer.eos_token_id\n                correct+=int(matched); capped+=int(hit_cap)\n                debug_record(\'accuracy_sample\',dict(mode=mode,arm=arm,question_id=i,gold=row[\'gold\'],\n                    text=result.texts[0],tokens=list(result.token_ids[0]),correct=bool(matched),hit_cap=bool(hit_cap)))\n                if (i+1)%256==0: print(f\'Accuracy {mode}/{arm}: {i+1}/{len(test_rows)}\',flush=True)\n            accuracy_results[mode,arm]=dict(correct=correct,total=len(test_rows),accuracy=correct/len(test_rows),capped=capped)\n            debug_record(\'accuracy_result\',dict(mode=mode,arm=arm,**accuracy_results[mode,arm]))\nprint(f\'{DATASET}: {len(test_rows)} questions, numeric exact match:\')\nfor mode,label in [(\'explicit_cot\',\'Explicit\'),(\'codi\',\'CODI\')]:\n    if mode not in MODES: continue\n    before,after=accuracy_results[mode,\'dense\'],accuracy_results[mode,\'rank96\']\n    print(f"{label}: dense {before[\'accuracy\']:.2%} ({before[\'correct\']}/{before[\'total\']}) -> "\n          f"rank96 {after[\'accuracy\']:.2%} ({after[\'correct\']}/{after[\'total\']}); "\n          f"change {100*(after[\'accuracy\']-before[\'accuracy\']):+.2f} percentage points.")\n    if before[\'capped\'] or after[\'capped\']:\n        print(f"  Reached token cap: dense {before[\'capped\']}; rank96 {after[\'capped\']}. Scored with the same answer extractor.")\n'

Each backbone is loaded and fitted once, then evaluated on all three datasets.
Fitted heads and completed pairs are cached under the experiment fingerprint, so
rerunning in the same Kaggle working directory reuses completed work. Downloaded base
weights, cached states and head fitting are outside inference latency measurements.

In [ ]:
def pair_reports(scope):
    tables={}
    for per_token in (False,True):
        for arm in ('dense','rank96'):
            samples=[s for s in scope['profile_samples'] if s['arm']==arm]
            report=scope['runtime'].bottleneck_means(samples,per_token=per_token)
            table=pd.DataFrame([values for _,values in report],index=[name for name,_ in report],columns=runtime.COLUMNS)
            table.index.name='Mean ms / token or latent step' if per_token else 'Mean ms / question'
            for col in table:
                if table[col].notna().any():
                    assert abs(table.iloc[1:-1][col].sum()-table.iloc[0][col])<.05
            tables[arm+('_per_token' if per_token else '_per_question')]=table
    summaries=[]
    for mode in scope['MODES']:
        for arm in ('dense','rank96'):
            clean=[r for r in scope['clean_samples'] if r['mode']==mode and r['arm']==arm]
            profiled=[r for r in scope['profile_samples'] if r['mode']==mode and r['arm']==arm]
            mean=sum(r['ms'] for r in clean)/len(clean)
            summaries.append(dict(mode=mode,arm=arm,mean_ms=mean,
                mean_visible_tokens=sum(r['visible_tokens'] for r in clean)/len(clean),
                profiler_inflation=(sum(r['total_ms'] for r in profiled)/len(profiled))/mean,
                **scope['accuracy_results'][mode,arm]))
    return tables,summaries

def show_pair(key, result, opened=False):
    title=f"{key[0]} / {key[1]} - four timing tables"
    pieces=[f"<details {'open' if opened else ''}><summary><b>{html.escape(title)}</b></summary>",
        '<p>Profiled elapsed times include instrumentation; use the clean means for speed claims.</p>']
    if key[0]!='gpt2': pieces.append('<p>CODI: N/A — no trained CODI checkpoint for this backbone.</p>')
    for name,data in result['tables'].items():
        table=pd.DataFrame(data['data'],index=data['index'],columns=data['columns'],dtype=float)
        per_token=name.endswith('_per_token')
        label=('Before: normal dense head' if name.startswith('dense') else 'After: rank-96 global head')
        pieces.append(f'<p><b>{label} — '+('ms/token or latent step' if per_token else 'ms/question')+'</b></p>')
        if not per_token and table['CODI overall'].notna().any():
            totals=table.iloc[0]
            shared=totals['CODI overall']-totals['CODI latent only']-totals['CODI visible only']
            pieces.append(f"<p>CODI {totals['CODI overall']:.3f} = {shared:.3f} shared + {totals['CODI latent only']:.3f} latent + {totals['CODI visible only']:.3f} visible.</p>")
        pieces.append(table.to_html(float_format=lambda value:f'{value:.3f}',na_rep='N/A',border=0))
    pieces.append('</details>')
    display(HTML('\n'.join(pieces)))

def safe_tables(tables):
    # Strict JSON: absent CODI cells are null, never fabricated zero measurements.
    return {key:json.loads(frame.to_json(orient='split')) for key,frame in tables.items()}

In [ ]:
for model_spec in MODEL_SPECS:
    model_key=model_spec['key']
    model_dir=EXPERIMENT_DIR/model_key
    model_dir.mkdir(exist_ok=True)
    print('\n'+model_spec['label'],flush=True)
    completed={name:model_dir/name/'results.json' for name in ablation.DATASETS}
    if all(path.exists() for path in completed.values()):
        for name,path in completed.items(): all_results[model_key,name]=json.loads(path.read_text())
        print('Reusing all three completed dataset results.')
        continue
    scope=globals().copy()
    scope.update(MODEL_KEY=model_key,MODEL_SPEC=model_spec,RUN_DIR=model_dir,splits=dict(base_splits),
        MODES=['codi','explicit_cot'] if model_key=='gpt2' else ['explicit_cot'])
    scope['runtime']=SimpleNamespace(**{k:v for k,v in vars(REFERENCE_RUNTIME).items() if not k.startswith('__')})
    scope['runtime'].decode=ablation.decode
    scope['runtime'].profile_question=ablation.profile_question
    # Tag every raw event without changing the original timing/fitting loops.
    def model_debug(kind,payload,_key=model_key):
        debug_record(kind,dict(model=_key,payload=payload))
    scope['debug_record']=model_debug
    torch.manual_seed(SEED); random.seed(SEED)
    if model_key=='gpt2':
        exec(GPT2_LOAD,scope)
    else:
        tokenizer=AutoTokenizer.from_pretrained(model_spec['repo'],revision=model_spec['revision'])
        if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
        tokenizer.padding_side='left'
        backbone=AutoModelForCausalLM.from_pretrained(model_spec['repo'],revision=model_spec['revision'],
            torch_dtype=torch.float32,attn_implementation='sdpa')
        model=ablation.ExplicitBackbone(backbone).requires_grad_(False).to(device).eval()
        base=official_codi_base_model(model); full_head=base.get_output_embeddings()
        scope.update(model=model,tokenizer=tokenizer,base=base,full_head=full_head,
            weight=full_head.weight.detach(),bias=None if getattr(full_head,'bias',None) is None else full_head.bias.detach())
        debug_record('backbone',dict(model=model_key,repo=model_spec['repo'],revision=model_spec['revision'],
            parameters=sum(p.numel() for p in model.parameters()),hidden_size=model.config.hidden_size,
            vocabulary_size=model.eot_id,codi_available=False,math_finetuned=False))
        del model,tokenizer,base,full_head,backbone
    n_layers=ablation.block_roots(scope['model'])[1]
    scope['runtime'].bottleneck_means=partial(ablation.bottleneck_means,n_layers=n_layers)
    exec(PARITY_CHECK,scope)
    exec(FIT_HEADS,scope)
    for dataset in ablation.DATASETS:
        pair_dir=model_dir/dataset
        pair_dir.mkdir(exist_ok=True)
        if completed[dataset].exists():
            all_results[model_key,dataset]=json.loads(completed[dataset].read_text())
            print('Reuse completed pair:',model_key,dataset)
            continue
        print(f'\n{model_spec["label"]} / {dataset}',flush=True)
        scope['DATASET']=dataset
        scope['splits']=dict(base_splits,timing=ablation.timing_questions(dataset,base_splits,evaluation,SEED))
        scope['test_rows']=evaluation[dataset]
        def pair_debug(kind,payload,_model=model_key,_dataset=dataset):
            debug_record(kind,dict(model=_model,dataset=_dataset,payload=payload))
        scope['debug_record']=pair_debug
        exec(TIMING_RUN,scope)
        exec(ACCURACY_RUN,scope)
        tables,summaries=pair_reports(scope)
        result=dict(model=model_key,dataset=dataset,codi_available='codi' in scope['MODES'],
            summaries=summaries,tables=safe_tables(tables),clean_samples=scope['clean_samples'],
            profile_samples=scope['profile_samples'])
        save_json(completed[dataset],result)
        all_results[model_key,dataset]=result
        for mode in scope['MODES']:
            before=next(r for r in summaries if r['mode']==mode and r['arm']=='dense')
            after=next(r for r in summaries if r['mode']==mode and r['arm']=='rank96')
            print(f"{mode}, WITHOUT profiler: {before['mean_ms']:.2f} -> {after['mean_ms']:.2f} ms/question; "
                  f"visible tokens {before['mean_visible_tokens']:.1f} -> {after['mean_visible_tokens']:.1f}; "
                  f"profiler inflation {before['profiler_inflation']:.2f}x -> {after['profiler_inflation']:.2f}x.")
        del tables,summaries,result
    del scope
    gc.collect(); torch.cuda.empty_cache()
print('Finished all nine model/dataset pairs.')

Results below retain four columns and four tables per model/dataset pair. Expand a
panel to inspect it. Total rows repeat at the top and bottom; only means are displayed.
CODI overall includes shared prompt preparation/prefill. Per-token denominators are
visible tokens for explicit/visible, six latent steps for latent, and latent + visible
for overall. Columns with different denominators cannot be added.

Generated lengths may change after compression. Clean latency comparisons include
that effect; the accuracy results reveal changes in answer quality. Token counts from
different model tokenizers are different units, so use ms/question for model comparisons.

In [ ]:
summary_rows=[]
for spec in MODEL_SPECS:
    for dataset in ablation.DATASETS:
        result=all_results[spec['key'],dataset]
        for mode in ('explicit_cot','codi'):
            rows=[r for r in result['summaries'] if r['mode']==mode]
            if not rows: continue
            before=next(r for r in rows if r['arm']=='dense')
            after=next(r for r in rows if r['arm']=='rank96')
            summary_rows.append(dict(Model=spec['label'],Dataset=dataset,Mode=mode,
                **{'Mean ms: dense -> rank96':f"{before['mean_ms']:.2f} -> {after['mean_ms']:.2f}",
                   'Accuracy: dense -> rank96':f"{before['accuracy']:.2%} -> {after['accuracy']:.2%}"}))
display(pd.DataFrame(summary_rows))
for spec in MODEL_SPECS:
    for dataset in ablation.DATASETS:
        key=spec['key'],dataset
        show_pair(key,all_results[key],opened=key==('gpt2','gsm8k'))
print('All tables and individual timings: all_results[(model_key, dataset)].')
print('Raw events, predictions, setup and exclusions:',DEBUG_PATH)
print('Output folder:',EXPERIMENT_DIR)

Example inspection: `all_results['qwen2_5_0_5b','svamp']['clean_samples']`.
The single `debug.jsonl.gz` retains individual component events and predicted/gold answers.
Each completed pair also has one `results.json` checkpoint for reuse; opening these files
is optional. Kaggle session resets require preserving working outputs to retain caches.

Dataset sources: [GSM8K](https://github.com/openai/grade-school-math),
[SVAMP](https://github.com/arkilpatel/SVAMP),
[ASDiv](https://github.com/chaochun/nlu-asdiv-dataset). ASDiv numeric subset accuracy
is not full-ASDiv accuracy or the ASDiv-A cross-validation benchmark.